# GemmaFit v5 E2B Evidence Router Training

Colab-resumable notebook for the v5 E2B evidence router. The model learns compact evidence packet -> one bounded function call. It does not learn raw video, raw skeleton math, force, GRF, EMG, heart-rate status, fall-risk prediction, clinical labels, or medical conclusions.


In [1]:
# Phase -1 / 0: Resume diagnostic, Drive mount, paths, and atomic state helpers
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

DRIVE_ROOT = Path(os.environ.get('GEMMAFIT_DRIVE_ROOT', '/content/drive/MyDrive/GemmaFit_train'))
REPO_DIR = Path(os.environ.get('GEMMAFIT_REPO_DIR', '/content/GemmaFit'))
GEMMAFIT_GIT_REF = os.environ.get('GEMMAFIT_GIT_REF', 'codex/e2b-video-capability-probes')
DATASET_VARIANT = os.environ.get('DATASET_VARIANT', 'v5_1_hard')
USE_HARD_CASES = DATASET_VARIANT in {'v5_1', 'v5_1_hard', 'hard'}
RUN_SUFFIX = 'gemmafit_v5_1_e2b_evidence_router_hard_cases' if USE_HARD_CASES else 'gemmafit_v5_e2b_evidence_router'
BASE_MODEL_ID = os.environ.get('BASE_MODEL_ID', 'google/gemma-4-E2B-it')
MODEL_TAG = BASE_MODEL_ID.replace('/', '_').replace('-', '_')
RUN_NAME = f'{MODEL_TAG}_{RUN_SUFFIX}'
TRAINED = DRIVE_ROOT / 'trained_outputs'
CKPT_DIR = TRAINED / 'checkpoints' / RUN_NAME
ADAPTER_DIR = TRAINED / 'adapter' / RUN_NAME
MERGED_DIR = TRAINED / 'merged_fp16' / RUN_NAME
PHASE_DIR = TRAINED / f'PHASE_DONE_{RUN_NAME}'
RUN_STATE = TRAINED / f'RUN_STATE_{RUN_NAME}.json'
RUN_EVENTS = TRAINED / f'RUN_EVENTS_{RUN_NAME}.jsonl'
DISCONNECT_POINTS = TRAINED / f'DISCONNECT_POINTS_{RUN_NAME}.jsonl'
DONE_MARKER = TRAINED / f'TRAINING_DONE_{RUN_NAME}.json'
DATASET_REPO_PATH = Path('finetune/data/gemmafit_v5_1_e2b_evidence_router.json' if USE_HARD_CASES else 'finetune/data/gemmafit_v5_e2b_evidence_router.json')
DATASET_TRAIN_COUNT = int(os.environ.get('DATASET_TRAIN_COUNT', '50000' if USE_HARD_CASES else '8000'))
DATASET_VALIDATION_COUNT = int(os.environ.get('DATASET_VALIDATION_COUNT', '6000' if USE_HARD_CASES else '1200'))
DATASET_ZH_TW_RATIO = float(os.environ.get('DATASET_ZH_TW_RATIO', '0.45' if USE_HARD_CASES else '0.0'))
DATASET_SCHEMA_FUZZ_RATIO = float(os.environ.get('DATASET_SCHEMA_FUZZ_RATIO', '0.25')) if USE_HARD_CASES else None
DATASET_PATH = REPO_DIR / DATASET_REPO_PATH
METRICS_DIR = REPO_DIR / 'finetune' / 'metrics'
TOOL_EVAL = METRICS_DIR / 'tool_call_eval_v5_e2b.json'
REFUSAL_EVAL = METRICS_DIR / 'refusal_eval_v5_e2b.json'
ADVERSARIAL_EVAL = METRICS_DIR / 'adversarial_eval_v5_e2b.json'

for p in [TRAINED, CKPT_DIR, ADAPTER_DIR, MERGED_DIR, PHASE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def now_iso() -> str:
    return time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

def atomic_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass
    tmp.replace(path)

def record_event(phase: str, event: str, **kwargs) -> None:
    RUN_EVENTS.parent.mkdir(parents=True, exist_ok=True)
    payload = {'ts': now_iso(), 'run_name': RUN_NAME, 'phase': phase, 'event': event, **kwargs}
    with RUN_EVENTS.open('a', encoding='utf-8') as f:
        f.write(json.dumps(payload, ensure_ascii=False) + '\n')
    if event in {'dataset_validated', 'trainer_initialized', 'checkpoint_saved', 'adapter_saved', 'merged_hf_saved', 'eval_done', 'done_marker_written'}:
        with DISCONNECT_POINTS.open('a', encoding='utf-8') as f:
            f.write(json.dumps({**payload, 'note': 'safe_resume_point'}, ensure_ascii=False) + '\n')

def write_run_state(phase: str, **kwargs) -> None:
    state = {'version': 'v5_e2b_evidence_router', 'run_name': RUN_NAME, 'run_suffix': RUN_SUFFIX, 'phase': phase, 'updated_at': now_iso(), **kwargs}
    atomic_json(RUN_STATE, state)

def mark_phase_done(phase_id: str, outputs: list[str] | None = None, next_phase: str = '') -> None:
    payload = {'phase': phase_id, 'status': 'done', 'timestamp': now_iso(), 'run_name': RUN_NAME, 'outputs': outputs or [], 'next_phase': next_phase}
    atomic_json(PHASE_DIR / f'{phase_id}.json', payload)
    write_run_state(phase_id, last_completed_phase=phase_id, next_phase=next_phase)

def latest_checkpoint(path: Path) -> str | None:
    checkpoints = []
    if path.exists():
        for child in path.glob('checkpoint-*'):
            if child.is_dir():
                try:
                    checkpoints.append((int(child.name.split('-')[-1]), child))
                except ValueError:
                    pass
    return str(sorted(checkpoints)[-1][1]) if checkpoints else None

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

print('=== Resume diagnostic ===')
print('Run name:', RUN_NAME)
print('Base model:', BASE_MODEL_ID)
print('Done marker:', DONE_MARKER, DONE_MARKER.exists())
print('Last completed phase:', sorted([p.stem for p in PHASE_DIR.glob('*.json')])[-1:] or 'none')
print('Latest checkpoint:', latest_checkpoint(CKPT_DIR))
print('Repo dir:', REPO_DIR)
print('Dataset path:', DATASET_PATH)
print('Artifacts found:', [str(p) for p in [RUN_STATE, RUN_EVENTS, DONE_MARKER] if p.exists()])
record_event('0_paths', 'diagnostic_printed')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Resume diagnostic ===
Run name: google_gemma_4_E2B_it_gemmafit_v5_1_e2b_evidence_router_hard_cases
Base model: google/gemma-4-E2B-it
Done marker: /content/drive/MyDrive/GemmaFit_train/trained_outputs/TRAINING_DONE_google_gemma_4_E2B_it_gemmafit_v5_1_e2b_evidence_router_hard_cases.json False
Last completed phase: ['4_model_preflight']
Latest checkpoint: None
Repo dir: /content/GemmaFit
Dataset path: /content/GemmaFit/finetune/data/gemmafit_v5_1_e2b_evidence_router.json
Artifacts found: ['/content/drive/MyDrive/GemmaFit_train/trained_outputs/RUN_STATE_google_gemma_4_E2B_it_gemmafit_v5_1_e2b_evidence_router_hard_cases.json', '/content/drive/MyDrive/GemmaFit_train/trained_outputs/RUN_EVENTS_google_gemma_4_E2B_it_gemmafit_v5_1_e2b_evidence_router_hard_cases.jsonl']


In [2]:
# Phase 1: Install dependencies and load HF token
# Set INSTALL_DEPS=0 to skip when the runtime already has compatible packages.
INSTALL_DEPS = os.environ.get('INSTALL_DEPS', '1') == '1'
if INSTALL_DEPS:
    cmd = [
        sys.executable, '-m', 'pip', 'install', '-U',
        'transformers', 'datasets', 'accelerate', 'peft', 'trl', 'safetensors', 'huggingface_hub'
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping dependency install because INSTALL_DEPS=0')

HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('HF token loaded from environment')
    except Exception as exc:
        print('HF login skipped/failed:', exc)
else:
    print('No HF_TOKEN in environment. Public models may still load; gated models will fail.')
mark_phase_done('1_deps', ['transformers', 'datasets', 'peft', 'trl'], '2_repo_dataset')


Running: /usr/bin/python3 -m pip install -U transformers datasets accelerate peft trl safetensors huggingface_hub
No HF_TOKEN in environment. Public models may still load; gated models will fail.


In [3]:
# Phase 2 / 3: Repo checkout, v5 dataset generation, strict schema gates
if not REPO_DIR.exists():
    REPO_URL = os.environ.get('GEMMAFIT_REPO_URL', 'https://github.com/bkttt0429/GemmaFit.git')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Using existing repo:', REPO_DIR)

os.chdir(REPO_DIR)
if GEMMAFIT_GIT_REF:
    print('Checking out repo ref:', GEMMAFIT_GIT_REF)
    subprocess.run(['git', 'fetch', 'origin', GEMMAFIT_GIT_REF], check=True)
    subprocess.run(['git', 'checkout', '-B', GEMMAFIT_GIT_REF, f'origin/{GEMMAFIT_GIT_REF}'], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', GEMMAFIT_GIT_REF], check=True)
else:
    print('GEMMAFIT_GIT_REF is empty; using repo default checkout.')
subprocess.run([sys.executable, 'finetune/data/generate_v5_e2b_evidence_router.py', '--help'], check=True)
generate_cmd = [
    sys.executable,
    'finetune/data/generate_v5_e2b_evidence_router.py',
    '--train-count', str(DATASET_TRAIN_COUNT),
    '--validation-count', str(DATASET_VALIDATION_COUNT),
    '--out', str(DATASET_REPO_PATH),
    '--validate',
]
if USE_HARD_CASES:
    generate_cmd.extend(['--hard-cases', '--zh-tw-ratio', str(DATASET_ZH_TW_RATIO)])
    if DATASET_SCHEMA_FUZZ_RATIO is not None:
        generate_cmd.extend(['--schema-fuzz-ratio', str(DATASET_SCHEMA_FUZZ_RATIO)])
print('Dataset generation command:', ' '.join(generate_cmd))
subprocess.run(generate_cmd, check=True)
subprocess.run([
    sys.executable,
    'finetune/eval_v5_e2b_evidence_router.py',
    '--dataset', str(DATASET_REPO_PATH),
    '--strict',
], check=True)
DATASET_SHA256 = sha256_file(DATASET_PATH)
with DATASET_PATH.open('r', encoding='utf-8') as f:
    dataset_payload = json.load(f)
TRAIN_ROWS = len(dataset_payload['train'])
VALIDATION_ROWS = sum(len(v) for v in dataset_payload['validation'].values())
print({'dataset_sha256': DATASET_SHA256, 'train_rows': TRAIN_ROWS, 'validation_rows': VALIDATION_ROWS})
record_event('3_dataset', 'dataset_validated', dataset=str(DATASET_PATH), sha256=DATASET_SHA256, train_rows=TRAIN_ROWS, validation_rows=VALIDATION_ROWS)
mark_phase_done('3_dataset_validated', [str(DATASET_PATH), str(TOOL_EVAL)], '4_model_preflight')


Using existing repo: /content/GemmaFit
Checking out repo ref: codex/e2b-video-capability-probes
Dataset generation command: /usr/bin/python3 finetune/data/generate_v5_e2b_evidence_router.py --train-count 50000 --validation-count 6000 --out finetune/data/gemmafit_v5_1_e2b_evidence_router.json --validate --hard-cases --zh-tw-ratio 0.45 --schema-fuzz-ratio 0.25
{'dataset_sha256': '14275d70f1d15927cb1b1f7ae1bf19677fd155b5e9c5601fd5909c0d8d8f4ddc', 'train_rows': 50000, 'validation_rows': 6000}


In [4]:
# Phase 4: Model/tokenizer preflight and dataset preview
from datasets import Dataset
from transformers import AutoTokenizer

with DATASET_PATH.open('r', encoding='utf-8') as f:
    data = json.load(f)

print('Validation splits:', {k: len(v) for k, v in data['validation'].items()})
print('First row type:', data['train'][0].get('row_type'))
print('First assistant target:', data['train'][0]['messages'][-1]['content'][:500])

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_messages(row: dict) -> str:
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)
    return '\n'.join(f"{m['role']}: {m['content']}" for m in row['messages'])

lengths = [len(tokenizer(render_messages(row)).input_ids) for row in data['train'][:256]]
print({'max_sample_tokens': max(lengths), 'avg_sample_tokens': sum(lengths) / len(lengths)})
mark_phase_done('4_model_preflight', [BASE_MODEL_ID], '5_lora_setup')


Validation splits: {'care_log': 400, 'persona_report': 400, 'dual_task': 400, 'realtime_event': 400, 'activity_uncertain': 400, 'unsupported': 400, 'adversarial': 400, 'zh_tw': 400, 'schema_fuzz': 400, 'tracking_uncertainty': 400, 'parent_task_uncertain': 400, 'sub_action_fallback': 400, 'conflicting_evidence': 400, 'memory_policy': 400, 'unsupported_zh_tw': 400}
First row type: tracking_uncertainty
First assistant target: {"function":"refuse_unsupported_question","args":{"reason":"insufficient_evidence","safe_alternative":"Tracking is predicted for this window, so hard coaching is blocked. Please use a clearer single-person view before feedback.","selection_basis":"Person tracking does not allow hard judgment for this event packet.","evidence_refs":["tracking.judgment.blocked"],"refusal_level":3}}
{'max_sample_tokens': 1505, 'avg_sample_tokens': 1198.6328125}


In [5]:
import subprocess
import sys
import os
# Phase 5 / 6: LoRA SFT training with checkpoint resume
# Fix: Install bitsandbytes if not already installed.
try:
    import bitsandbytes
except ImportError:
    print("bitsandbytes not found, installing...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'bitsandbytes>=0.46.1'], check=True)
    print("bitsandbytes installed.")
    import bitsandbytes

import importlib
importlib.invalidate_caches()
if 'transformers' in sys.modules:
    import transformers.utils.import_utils
    transformers.utils.import_utils._bitsandbytes_available = True
    print("Forced transformers to recognize bitsandbytes.")

from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
import torch

MAX_SEQ_LENGTH = int(os.environ.get('MAX_SEQ_LENGTH', '2048'))
TRAIN_LIMIT = int(os.environ.get('TRAIN_LIMIT', '0'))
EVAL_LIMIT = int(os.environ.get('EVAL_LIMIT', '256'))
OUTPUT_DIR = str(CKPT_DIR)

train_rows = data['train'][:TRAIN_LIMIT or None]
eval_pool = []
for split_rows in data['validation'].values():
    eval_pool.extend(split_rows)
eval_rows = eval_pool[:EVAL_LIMIT]

def to_text_dataset(rows: list[dict]) -> Dataset:
    return Dataset.from_list([{'text': render_messages(row), 'row_type': row.get('row_type', '')} for row in rows])

train_dataset = to_text_dataset(train_rows)
eval_dataset = to_text_dataset(eval_rows)

use_4bit = os.environ.get('LOAD_IN_4BIT', '1') == '1'
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type='nf4') if use_4bit else None
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    quantization_config=quant_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    trust_remote_code=False,
)
if use_4bit:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=int(os.environ.get('LORA_R', '16')),
    lora_alpha=int(os.environ.get('LORA_ALPHA', '32')),
    lora_dropout=float(os.environ.get('LORA_DROPOUT', '0.05')),
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
record_event('6_training', 'trainer_initialized', checkpoint_dir=str(CKPT_DIR))

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=int(os.environ.get('BATCH_SIZE', '1')),
    gradient_accumulation_steps=int(os.environ.get('GRAD_ACCUM', '8')),
    learning_rate=float(os.environ.get('LEARNING_RATE', '2e-4')),
    num_train_epochs=float(os.environ.get('NUM_TRAIN_EPOCHS', '1')),
    logging_steps=10,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    bf16=torch.cuda.is_available(),
    report_to=[],
    dataset_text_field='text',
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)
resume = latest_checkpoint(CKPT_DIR)
print('Resuming from:', resume)
trainer.train(resume_from_checkpoint=resume)
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
record_event('6_training', 'adapter_saved', adapter=str(ADAPTER_DIR), latest_checkpoint=latest_checkpoint(CKPT_DIR))
mark_phase_done('7_adapter_saved', [str(ADAPTER_DIR)], '8_merge')


Forced transformers to recognize bitsandbytes.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

ValueError: Target module Gemma4ClippableLinear(
  (linear): Linear4bit(in_features=768, out_features=768, bias=False)
) is not supported. Currently, only the following modules are supported: `torch.nn.Linear`, `torch.nn.Embedding`, `torch.nn.Conv1d`, `torch.nn.Conv2d`, `torch.nn.Conv3d`, `transformers.pytorch_utils.Conv1D`, `torch.nn.MultiheadAttention.`.

In [ ]:
# Phase 8: Merge LoRA adapter into HF/safetensors export
from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, torch_dtype=torch.float16, device_map='auto', trust_remote_code=False)
merged = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
merged = merged.merge_and_unload()
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
record_event('8_merge', 'merged_hf_saved', merged_hf=str(MERGED_DIR))
mark_phase_done('8_merged_hf_saved', [str(MERGED_DIR)], '9_generation_smoke')


In [ ]:
# Phase 9: Generation smoke on held-out rows
import json
import re
from transformers import pipeline

SMOKE_LIMIT = int(os.environ.get('SMOKE_LIMIT', '8'))
text_gen = pipeline('text-generation', model=str(MERGED_DIR), tokenizer=tokenizer, device_map='auto')
smoke_rows = eval_pool[:SMOKE_LIMIT]
smoke_results = []
for idx, row in enumerate(smoke_rows):
    prompt_messages = row['messages'][:-1]
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = '\n'.join(f"{m['role']}: {m['content']}" for m in prompt_messages) + '\nassistant:'
    out = text_gen(prompt, max_new_tokens=256, do_sample=False, return_full_text=False)[0]['generated_text']
    start, end = out.find('{'), out.rfind('}')
    parsed = None
    if 0 <= start < end:
        try:
            parsed = json.loads(out[start:end + 1])
        except Exception:
            parsed = None
    smoke_results.append({'index': idx, 'row_type': row.get('row_type'), 'expected': row.get('expected_function'), 'parsed': parsed, 'raw': out[-1000:]})
print(json.dumps(smoke_results[:3], indent=2, ensure_ascii=False))
record_event('9_smoke', 'generation_smoke_done', rows=len(smoke_results))
mark_phase_done('9_generation_smoke_done', [], '10_handoff')


In [ ]:
# Phase 10: Optional LiteRT conversion / artifact handoff note
# Official LiteRT conversion is intentionally a separate step because package
# compatibility can differ from the training runtime. Use the conversion-only
# notebook or a clean runtime after MERGED_DIR exists.
LITERT_TARGET = DRIVE_ROOT / 'gemmafit-v5-e2b-evidence-router.litertlm'
print('Merged HF export for conversion:', MERGED_DIR)
print('Expected final .litertlm    :', LITERT_TARGET)
print('Local finalize command after download:')
print('python finetune/prepare_litert_artifact.py --profile v5-e2b --source-litertlm path/to/gemmafit-v5-e2b-evidence-router.litertlm --run-smoke')
mark_phase_done('10_handoff_ready', [str(MERGED_DIR), str(LITERT_TARGET)], '11_done')


In [ ]:
# Phase 11: Eval reports and done marker
subprocess.run([
    sys.executable,
    'finetune/eval_v5_e2b_evidence_router.py',
    '--dataset', str(DATASET_REPO_PATH),
    '--output', str(TOOL_EVAL),
    '--refusal-output', str(REFUSAL_EVAL),
    '--adversarial-output', str(ADVERSARIAL_EVAL),
    '--strict',
], check=True)
trainer_state = CKPT_DIR / 'trainer_state.json'
done = {
    'version': 'v5_1_e2b_evidence_router_hard_cases' if USE_HARD_CASES else 'v5_e2b_evidence_router',
    'run_name': RUN_NAME,
    'run_suffix': RUN_SUFFIX,
    'finished_at': now_iso(),
    'status': 'training_complete',
    'last_completed_phase': '11_eval_done',
    'dataset_variant': DATASET_VARIANT,
    'hard_cases': USE_HARD_CASES,
    'dataset_path': str(DATASET_REPO_PATH),
    'dataset_sha256': DATASET_SHA256,
    'train_rows': TRAIN_ROWS,
    'validation_rows': VALIDATION_ROWS,
    'model': BASE_MODEL_ID,
    'checkpoint_dir': str(CKPT_DIR),
    'best_checkpoint': latest_checkpoint(CKPT_DIR),
    'adapter_path': str(ADAPTER_DIR),
    'merged_hf_path': str(MERGED_DIR),
    'litertlm_path': 'models/gemmafit-v5-e2b-evidence-router.litertlm',
    'conversion_status': 'not_started',
    'tool_call_eval': str(TOOL_EVAL),
    'refusal_eval': str(REFUSAL_EVAL),
    'adversarial_eval': str(ADVERSARIAL_EVAL),
    'resume_log': str(RUN_EVENTS),
    'disconnect_points': str(DISCONNECT_POINTS),
}
atomic_json(DONE_MARKER, done)
record_event('11_done', 'done_marker_written', done_marker=str(DONE_MARKER))
mark_phase_done('11_eval_done', [str(DONE_MARKER), str(TOOL_EVAL)], '')
print(json.dumps(done, indent=2, ensure_ascii=False))
